# Custom RAG Pipeline for 2026 Car Recommendation Chatbot

This Jupyter Notebook implements an end-to-end **Retrieval-Augmented Generation (RAG)** pipeline for an automotive recommendation assistant. It ingests a CSV dataset of vehicles, processes and chunks vehicle specifications into rich semantic documents, computes vector embeddings using Google GenAI (`gemini-embedding-2-preview`), performs hybrid cosine + lexical matching, and generates grounded 2026 car recommendations with `gemini-3.8-flash`.

### Key Architecture Steps:
1. **Environment Setup & Google GenAI SDK Installation** (`google-genai`)
2. **API Key Setup**: Secure retrieval via Google Colab `userdata` or environment variable
3. **CSV Ingestion & Validation**: Load, parse, and normalize vehicle catalog columns
4. **Semantic Document Chunking**: Transform tabular rows into domain-specific knowledge chunks (Overview, Specs, Efficiency, Safety & Tech, Practicality)
5. **Vector Indexing & Embeddings**: Generate 768-dim embeddings via `gemini-embedding-2-preview` with local caching
6. **Hybrid Retrieval Engine**: Combine vector cosine similarity with lexical BM25/attribute filters (budget, AWD, seating)
7. **Grounding & Chatbot Synthesis**: Prompt `gemini-3.8-flash` with strict anti-hallucination instructions to return verified car suggestions and structured metadata

In [ ]:
# Step 1: Install modern Google GenAI SDK and data analysis libraries
!pip install -q -U google-genai pandas numpy scikit-learn tabulate

In [ ]:
import os
import json
import re
import numpy as np
import pandas as pd
from typing import List, Dict, Any, Optional
from google import genai
from google.genai import types

# Step 2: Configure Gemini API Key
# In Google Colab, you can store your key in the 'Secrets' tab (🔑) under 'GEMINI_API_KEY'
# Or get your key from: https://aistudio.google.com/api-keys
try:
    from google.colab import userdata
    api_key = userdata.get('GEMINI_API_KEY')
except Exception:
    api_key = os.environ.get('GEMINI_API_KEY', '')

if not api_key:
    api_key = input('Please enter your Gemini API Key (or set GEMINI_API_KEY env var): ').strip()

client = genai.Client(api_key=api_key)
print('Google GenAI Client initialized successfully!')

## Step 3: Create / Load the 2026 Vehicle Inventory CSV

Here we provide sample vehicle inventory for 2026 models with electric, hybrid, SUV, sedan, and truck models. You can also upload your own custom CSV file.

In [ ]:
csv_content = """id,make,model,year,trim,bodyType,fuelType,price,leasePerMonth,mpgCity,mpgHwy,electricRangeMiles,horsepower,drivetrain,seatingCapacity,cargoVolumeCuFt,safetyRating,acceleration0to60,idealFor,description
toyota-rav4-hybrid-2026,Toyota,RAV4 Hybrid,2026,XSE AWD,SUV,Hybrid,38200,399,41,38,,225,AWD,5,69.8,5-Star NHTSA / IIHS Top Safety Pick,7.1 sec,"Daily commuters and active weekenders seeking unbeatable fuel economy and standard AWD","Benchmark compact hybrid crossover delivering 40+ combined MPG, rugged exterior, and legendary Toyota reliability."
tesla-model-y-2026,Tesla,Model Y,2026,Long Range AWD,Crossover,Electric,46990,459,,,320,390,AWD,5,76.2,5-Star NHTSA / IIHS Top Safety Pick+,4.6 sec,"Tech-forward drivers and road-trippers wanting effortless EV travel and high cargo space","The best-selling electric crossover with seamless Supercharger network access, brisk acceleration, and dual cargo trunks."
honda-cr-v-hybrid-2026,Honda,CR-V,2026,Sport-L Hybrid AWD,SUV,Hybrid,40250,410,40,34,,204,AWD,5,76.5,5-Star NHTSA / IIHS Top Safety Pick+,7.8 sec,"Small families and pet owners needing cavernous cargo room and plush ride comfort","Standard-bearer for family practicality, offering limousine-like rear legroom and class-leading cargo space."
hyundai-ioniq-5-2026,Hyundai,Ioniq 5,2026,Limited AWD,Crossover,Electric,53900,495,,,270,325,AWD,5,59.3,5-Star NHTSA / IIHS Top Safety Pick+,4.5 sec,"Design enthusiasts and EV adopters who value rapid DC fast-charging on road trips","Retro-futuristic electric crossover built on an 800V architecture enabling 10% to 80% charging in 18 minutes."
kia-telluride-2026,Kia,Telluride,2026,SX Prestige X-Line AWD,SUV,Gasoline,53450,560,19,25,,291,AWD,7,87.0,5-Star NHTSA / IIHS Top Safety Pick+,6.7 sec,"Growing families and road-trippers needing 3 comfortable rows of seating and towing capability","The reigning champion of 3-row family SUVs, combining luxurious appointments, true 7-adult seating, and 5,500 lbs towing."
honda-civic-hybrid-2026,Honda,Civic,2026,Sport Touring Hybrid,Sedan,Hybrid,32500,319,50,47,,200,FWD,5,14.8,5-Star NHTSA / IIHS Top Safety Pick+,6.8 sec,"Young professionals and daily commuters seeking 50 MPG, nimble handling, and upscale tech","High-mileage compact sport sedan delivering punchy electric-assist torque and 50 MPG without plug-in hassle."
bmw-330i-xdrive-2026,BMW,330i xDrive,2026,M Sport Package,Sedan,Gasoline,49200,539,24,33,,255,AWD,5,17.0,5-Star NHTSA / IIHS Top Safety Pick,5.4 sec,"Driving enthusiasts who crave spirited performance, German engineering, and all-weather AWD","The quintessential sports sedan delivering telepathic road feedback, impeccable balance, and a modern curved display."
subaru-outback-2026,Subaru,Outback,2026,Wilderness AWD,Wagon,Gasoline,41200,429,21,26,,260,AWD,5,75.6,5-Star NHTSA / IIHS Top Safety Pick+,5.8 sec,"Snow-belt residents, campers, and outdoor adventurers who need unstoppable winter prowess","Trail-ready adventure wagon elevated with 9.5-inch ground clearance, skid plates, all-terrain tires, and Symmetrical AWD."
toyota-prius-prime-2026,Toyota,Prius Prime,2026,XSE PHEV,Sedan,Plug-in Hybrid,36400,359,52,51,44,220,FWD,5,20.3,5-Star NHTSA / IIHS Top Safety Pick+,6.5 sec,"Smart commuters wanting electric daily driving without range anxiety and 50+ MPG road trips","Striking sports-hatch offering 44 miles of pure electric driving for daily commutes and 600+ miles total road trip range."
ford-f150-lightning-2026,Ford,F-150 Lightning,2026,Flash Extended Range,Truck,Electric,69995,719,,,320,580,AWD,5,52.8,5-Star NHTSA Overall Safety Rating,3.8 sec,"Contractors and homeowners who want zero gas emissions, massive mobile power, and supercar acceleration","All-electric pickup with 9.6 kW Pro Power mobile generator, Mega Power Frunk, and 7,700 lbs towing capacity."
"""

# Save sample CSV
with open('car_inventory_2026.csv', 'w') as f:
    f.write(csv_content)

# Load into Pandas DataFrame
df = pd.read_csv('car_inventory_2026.csv')
print(f'Loaded {len(df)} vehicles from CSV:')
df[['year', 'make', 'model', 'trim', 'fuelType', 'price', 'drivetrain']].head(5)

## Step 4: Semantic Chunking for Automotive RAG

Rather than indexing bare raw rows, we transform each vehicle into multi-aspect contextual documents: **Overview**, **Efficiency & Powertrain**, **Safety & Dimensions**, and **Practicality & Target Driver**. This enables precise semantic retrieval whether the user asks about snow traction, fuel economy, car seats, or budget.

In [ ]:
class CarDocumentChunk:
    def __init__(self, chunk_id: str, car_id: str, car_name: str, category: str, content: str, metadata: Dict[str, Any]):
        self.id = chunk_id
        self.car_id = car_id
        self.car_name = car_name
        self.category = category
        self.content = content
        self.metadata = metadata
        self.embedding = None

def create_car_chunks(row: pd.Series) -> List[CarDocumentChunk]:
    car_name = f"{int(row['year'])} {row['make']} {row['model']} {row['trim']}"
    car_id = str(row['id'])
    meta = row.to_dict()
    chunks = []

    # 1. Overview Chunk
    c1 = f"{car_name} is a {row['bodyType']} with starting MSRP of ${int(row['price']):,}. " \
         f"Powertrain: {row['fuelType']}, Drivetrain: {row['drivetrain']}, Power: {row['horsepower']} HP. " \
         f"Overview: {row['description']} Ideal for: {row['idealFor']}"
    chunks.append(CarDocumentChunk(f"{car_id}-overview", car_id, car_name, 'overview', c1, meta))

    # 2. Efficiency Chunk
    if row['fuelType'] == 'Electric':
        c2 = f"{car_name} Electric Performance: 100% pure electric with {row.get('electricRangeMiles', 300)} miles EPA range. " \
             f"{row['horsepower']} HP instant torque, 0-60 in {row['acceleration0to60']}. Zero gas emissions."
    else:
        c2 = f"{car_name} Fuel Efficiency: {row['mpgCity']} MPG City / {row['mpgHwy']} MPG Highway. " \
             f"Engine output: {row['horsepower']} HP, 0-60 in {row['acceleration0to60']}. Powertrain: {row['fuelType']}."
    chunks.append(CarDocumentChunk(f"{car_id}-efficiency", car_id, car_name, 'efficiency', c2, meta))

    # 3. Practicality & Safety Chunk
    c3 = f"{car_name} Dimensions & Safety: Seating for {int(row['seatingCapacity'])} passengers. " \
         f"Max cargo volume: {row['cargoVolumeCuFt']} cubic feet. Drivetrain: {row['drivetrain']}. " \
         f"Crash Test Rating: {row['safetyRating']}. Ideal for: {row['idealFor']}"
    chunks.append(CarDocumentChunk(f"{car_id}-practicality", car_id, car_name, 'practicality', c3, meta))

    return chunks

# Build knowledge chunks collection
all_chunks: List[CarDocumentChunk] = []
for _, row in df.iterrows():
    all_chunks.extend(create_car_chunks(row))

print(f'Generated {len(all_chunks)} semantic chunks from {len(df)} vehicles.')
print('Sample Chunk:', all_chunks[0].content)

## Step 5: Generate Vector Embeddings via Gemini API

We use `gemini-embedding-2-preview` to generate dense vector embeddings for our chunks. We also implement a cosine similarity calculation function.

In [ ]:
def get_embedding(text: str) -> Optional[List[float]]:
    try:
        resp = client.models.embed_content(
            model='gemini-embedding-2-preview',
            contents=text
        )
        if resp.embeddings and len(resp.embeddings) > 0:
            return resp.embeddings[0].values
        elif hasattr(resp, 'embedding') and resp.embedding:
            return resp.embedding.values
    except Exception as e:
        print(f'Embedding warning: {e}')
    return None

def cosine_similarity(a: List[float], b: List[float]) -> float:
    va = np.array(a)
    vb = np.array(b)
    norm_a = np.linalg.norm(va)
    norm_b = np.linalg.norm(vb)
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return float(np.dot(va, vb) / (norm_a * norm_b))

# Index initial chunks
print('Embedding knowledge chunks...')
embedded_count = 0
for c in all_chunks:
    emb = get_embedding(c.content)
    if emb:
        c.embedding = emb
        embedded_count += 1

print(f'Successfully embedded {embedded_count}/{len(all_chunks)} chunks into vector space.')

## Step 6: Hybrid Retrieval Engine (Semantic Vector + Domain Filtering)

Car buyers frequently search with composite intent (e.g. *"SUV with AWD under $42,000 for 5 people"*). Our hybrid retriever blends cosine semantic similarity with attribute filters for budget, seating, and winter driveability.

In [ ]:
def hybrid_retrieve(query: str, chunks: List[CarDocumentChunk], top_k: int = 4) -> List[Dict[str, Any]]:
    query_emb = get_embedding(query)
    q_lower = query.lower()
    results = []

    for chunk in chunks:
        # 1. Semantic Cosine Score
        semantic_score = 0.0
        if query_emb and chunk.embedding:
            semantic_score = max(0.0, cosine_similarity(query_emb, chunk.embedding))

        # 2. Lexical & Intent Boost
        content_lower = chunk.content.lower()
        terms = [t for t in re.findall(r'\w+', q_lower) if len(t) > 2]
        matches = sum(1 for t in terms if t in content_lower)
        lexical_score = matches / max(1, len(terms))

        intent_boost = 0.0
        car_price = chunk.metadata.get('price', 0)
        car_drivetrain = chunk.metadata.get('drivetrain', '')
        car_fuel = chunk.metadata.get('fuelType', '')

        # Budget parsing (e.g. 'under 40k' or 'under $45,000')
        budget_match = re.search(r'under\s*\$?(\d+)(?:k|000)?', q_lower)
        if budget_match:
            raw_budget = int(budget_match.group(1))
            max_budget = raw_budget * 1000 if raw_budget < 1000 else raw_budget
            if car_price <= max_budget:
                intent_boost += 0.25
            else:
                intent_boost -= 0.35

        # AWD / Snow requirement
        if any(w in q_lower for w in ['snow', 'winter', 'awd', 'all-wheel']):
            if car_drivetrain == 'AWD':
                intent_boost += 0.3

        # EV / Hybrid
        if 'electric' in q_lower or 'ev' in q_lower:
            if car_fuel == 'Electric':
                intent_boost += 0.3
        elif 'hybrid' in q_lower:
            if 'Hybrid' in car_fuel:
                intent_boost += 0.3

        # Blended final score
        final_score = (semantic_score * 0.6 + lexical_score * 0.2 + intent_boost * 0.2)
        results.append({
            'chunk': chunk,
            'score': round(float(final_score), 3),
            'car_id': chunk.car_id,
            'car_name': chunk.car_name
        })

    results.sort(key=lambda x: x['score'], reverse=True)
    return results[:top_k]

# Test retrieval with a realistic user query
test_query = "Reliable hybrid SUV under $40,000 with AWD for snowy winters"
retrieved = hybrid_retrieve(test_query, all_chunks, top_k=4)
print(f"Retrieval results for query: '{test_query}'\n")
for r in retrieved:
    print(f"[{r['score']}] {r['car_name']} ({r['chunk'].category})")
    print(f"       {r['chunk'].content[:120]}...\n")

## Step 7: Chatbot Synthesis with Gemini 3.8 Flash

Here we feed the retrieved context chunks directly to `gemini-3.8-flash`. The prompt instructs the model to:
- Ground recommendations in verified specs (pricing, MPG/range, safety, seating)
- Avoid artificial AI marketing fluff (anti-slop rule)
- Return structured recommendation cards and suggested follow-ups

In [ ]:
def recommend_cars(user_query: str, chat_history: List[Dict[str, str]] = None) -> str:
    # 1. Retrieve top chunks
    top_chunks = hybrid_retrieve(user_query, all_chunks, top_k=5)
    
    # Format context for Gemini
    context_text = "\n\n".join([
        f"[Vehicle Document: {r['car_name']} | Match Score: {r['score']}]\n{r['chunk'].content}"
        for r in top_chunks
    ])

    system_instruction = f"""You are CarMatch AI, an automotive advisor and car recommendation expert.
You recommend 2026 vehicles strictly grounded in the verified dealership inventory provided in the CONTEXT below.

VERIFIED SHOWROOM CONTEXT:
{context_text}

STRICT GUIDELINES:
1. Recommend only vehicles from the provided context.
2. Directly address the user's requirements (budget, commute, cargo, winter driving, tech).
3. Provide exact specifications: Starting MSRP, MPG or pure electric range, Horsepower, Drivetrain (AWD/FWD), and Safety rating.
4. Be objective, helpful, and concise. Do NOT use cheesy marketing hype (avoid 'game changer', 'unleash', 'supercharge').
5. Clearly note honest trade-offs (e.g. cabin quietness vs sportiness, charging vs gas stops).
6. Provide 2-3 concise follow-up questions at the end.
"""

    response = client.models.generate_content(
        model='gemini-3.8-flash',
        contents=user_query,
        config=types.GenerateContentConfig(
            system_instruction=system_instruction,
            temperature=0.5,
        )
    )
    return response.text

# Run recommendation query
query = "I drive 45 miles each day in cold weather, have a baby stroller, and my budget is $40,000. What car should I buy in 2026?"
print(f"USER: {query}\n")
recommendation = recommend_cars(query)
print(recommendation)

## Step 8: Upload and Ingest Your Own CSV

To test with your custom CSV file in Google Colab:
1. Click the Folder icon (📁) on the left sidebar
2. Upload your file (e.g. `my_cars.csv`)
3. Run the cell below to re-index all chunks and query your new inventory!

In [ ]:
def ingest_custom_csv(filepath: str):
    global all_chunks
    custom_df = pd.read_csv(filepath)
    print(f'Loading custom dataset: {len(custom_df)} rows')
    new_chunks = []
    for _, row in custom_df.iterrows():
        new_chunks.extend(create_car_chunks(row))
    
    print(f'Embedding {len(new_chunks)} custom chunks...')
    for c in new_chunks:
        c.embedding = get_embedding(c.content)
    
    all_chunks = new_chunks
    print('Custom CSV successfully indexed into live RAG pipeline!')

# Example usage:
# ingest_custom_csv('my_cars.csv')